# Produção de Energia no Brasil — Notebook de Análise
**Disciplina:** Linguagens de Programação  
**Professor:** Alexandre Neves Louzada  
**Aluno:** Dagner Costa Leal

## 1. Introdução ao problema
Como a produção de energia se distribui entre regiões, estados e fontes, e como ela evolui entre 2015 e 2024? Quais variáveis (consumo, capacidade, emissões, custo) se relacionam com a produção?

## 2. Explicação da base de dados
Base **simulada** do Tema 6 (`simulacao_producao_energia_brasil.csv`), fornecida pelo professor: 14.400 linhas e 13 colunas, com um registro mensal por UF e fonte.

| Coluna | Descrição |
|---|---|
| ano, mes, data | período do registro |
| regiao, uf | localização |
| fonte_energia | Hidrelétrica, Eólica, Solar, Termelétrica, Biomassa, Nuclear |
| producao_mwh, consumo_mwh | energia produzida e consumida (MWh) |
| capacidade_instalada | capacidade da usina/fonte |
| emissao_co2 | emissões de CO₂ |
| custo_medio_mwh | custo médio por MWh |
| percentual_renovavel | % renovável da fonte (0–100) |
| nivel_demanda | Baixo, Médio, Alto, Crítico |

## 3. Leitura dos dados

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
df = pd.read_csv("../dados/simulacao_producao_energia_brasil.csv", encoding="utf-8-sig")
print(df.shape)
print(df.dtypes)
df.head()

## 4. Limpeza e preparação

In [ ]:
print("Nulos:", df.isna().sum().sum(), "| Duplicadas:", df.duplicated().sum())
df = df.drop_duplicates()
df["data"] = pd.to_datetime(df["data"])
for c in ["regiao", "uf", "fonte_energia", "nivel_demanda"]:
    df[c] = df[c].astype(str).str.strip()
df.info()

## 5. Engenharia de atributos

In [ ]:
df["saldo_mwh"] = df["producao_mwh"] - df["consumo_mwh"]
df["utilizacao_capacidade"] = df["producao_mwh"] / df["capacidade_instalada"]
df["emissao_por_mwh"] = df["emissao_co2"] / df["producao_mwh"]
df["producao_renovavel_mwh"] = df["producao_mwh"] * df["percentual_renovavel"] / 100
df[["saldo_mwh", "utilizacao_capacidade", "emissao_por_mwh", "producao_renovavel_mwh"]].describe().T

## 6. Análise exploratória

In [ ]:
display(df.describe().T)
for c in ["regiao", "fonte_energia", "nivel_demanda"]:
    display(df.groupby(c)["producao_mwh"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False).round(0))

## 7. KPIs

In [ ]:
anual = df.groupby("ano")["producao_mwh"].sum()
kpis = {
    "Produção total (MWh)": df["producao_mwh"].sum(),
    "Consumo total (MWh)": df["consumo_mwh"].sum(),
    "Saldo (MWh)": df["saldo_mwh"].sum(),
    "Emissão total de CO2": df["emissao_co2"].sum(),
    "% renovável (ponderado pela produção)": df["producao_renovavel_mwh"].sum() / df["producao_mwh"].sum() * 100,
    "Custo médio por MWh": df["custo_medio_mwh"].mean(),
    "Variação 2015→2024 (%)": (anual.iloc[-1] / anual.iloc[0] - 1) * 100,
}
pd.Series(kpis).round(2)

## 8. Gráficos

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
s = df.groupby("fonte_energia")["producao_mwh"].sum().sort_values(ascending=False)
sns.barplot(x=s.values, y=s.index, ax=ax[0], color="#1d6fa5"); ax[0].set_title("Produção total por fonte (MWh)")
r = df.groupby("regiao")["producao_mwh"].sum().sort_values(ascending=False)
sns.barplot(x=r.values, y=r.index, ax=ax[1], color="#e6a117"); ax[1].set_title("Produção total por região (MWh)")
plt.tight_layout(); plt.savefig("../imagens/producao_fonte_regiao.png", dpi=120); plt.show()

In [ ]:
m = df.groupby("data")["producao_mwh"].sum()
plt.figure(figsize=(12, 4))
plt.plot(m, alpha=.4, label="Mensal")
plt.plot(m.rolling(12, min_periods=3).mean(), color="#d1495b", lw=2.5, label="Média móvel 12 meses")
plt.title("Produção mensal (MWh)"); plt.legend(); plt.tight_layout()
plt.savefig("../imagens/serie_temporal.png", dpi=120); plt.show()

In [ ]:
num = ["producao_mwh", "consumo_mwh", "capacidade_instalada", "emissao_co2", "custo_medio_mwh", "percentual_renovavel"]
plt.figure(figsize=(8, 5)); sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação de Pearson"); plt.tight_layout(); plt.savefig("../imagens/correlacao.png", dpi=120); plt.show()
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x="fonte_energia", y="emissao_co2", hue="fonte_energia", legend=False)
plt.title("Emissão de CO2 por fonte"); plt.show()

## 9. Interpretação dos resultados
A hidrelétrica é a fonte com maior produção total, seguida pela termelétrica. A produção total cresceu cerca de 45% entre 2015 e 2024. A produção tem correlação muito alta com consumo e capacidade instalada (≈ 0,98 e 1,00), o que é esperado em dados simulados e indica que a produção acompanha a capacidade. A correlação com emissões é moderada (≈ 0,21), concentrada na termelétrica. Confira estes números nos gráficos e tabelas gerados ao executar o notebook.

## 10. Conclusão
A matriz simulada é dominada por fontes renováveis, com a hidrelétrica à frente, e a produção cresce ao longo da década. As emissões de CO₂ vêm principalmente da termelétrica. Como a base é simulada, os resultados ilustram a metodologia e não a realidade do setor elétrico brasileiro.

---
Dagner Costa Leal · Linguagens de Programação · Prof. Alexandre Neves Louzada